In [ ]:
!pip install transformers accelerate sentencepiece torch
!pip install huggingface_hub

In [ ]:
from huggingface_hub import login

login()

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-2-7b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
import json
import os
import re
from pathlib import Path

BASE_DIR = Path("/content/dataCleaned")

LAWS_FOLDER = BASE_DIR / "Laws"
OUTPUT_FILE = BASE_DIR / "datasetTrain.json"

dataset = {"data": []}


def parse_examples(text):
    pattern = (
        r"Ejemplo\s*\d+:\s*\n\s*(?:Instrucción|Instruction):\s*(.*?)\n"
        r"\s*(?:Context|Contexto):\s*(.*?)\n"
        r"\s*(?:Respuesta|Response):\s*(.*?)(?=\n\s*Ejemplo\s*\d+:|\Z)"
    )

    matches = re.findall(pattern, text, re.DOTALL | re.IGNORECASE)

    data = []
    for inst, _ctx, resp in matches:
        data.append(
            {"instruction": inst.strip(), "context": "", "response": resp.strip()}
        )

    return {"data": data}


def validate_examples(parsed):
    valid_data = []

    for ex in parsed["data"]:
        if len(ex["instruction"]) < 10:
            continue
        if len(ex["response"]) < 30:
            continue
        if ex["context"] != "":
            continue

        valid_data.append(ex)

    return {"data": valid_data}


def generate_questions(context):
    prompt = f"""<s>[INST] <<SYS>>
You are a legal analyst specialized in Colombian law.

You generate training data for information distillation from Colombian legal documents.

You MUST follow all rules strictly. You are NOT allowed to invent information.
You MUST ONLY use the content of the provided document.
<</SYS>>

TASK

Read the Colombian legal document and generate EXACTLY TEN question-answer examples
designed to train a model in information distillation (extracting, compressing, and
reorganizing the essential content of a text).

Each example MUST include:
- instruction
- context
- response

The "context" field MUST ALWAYS be empty.

QUESTION TYPE DEFINITIONS AND MANDATORY DISTRIBUTION

1. idea_central : 2 examples
   The response must capture the core idea in a single sentence.

2. resumen_3_niveles : 1 example
   The response must provide:
   - Summary in 1 sentence
   - Summary in 3 sentences
   - 5 key bullet points

3. esencial_vs_accesorio : 1 example
   The response must classify content into essential and non-essential.

4. estructura_logica : 1 example
   The response must describe the logical structure of the document.

5. reescritura_simplificada : 1 example
   The response must be a plain-language rewrite accessible to a layperson.

6. intencion_autor : 1 example
   The response must identify the purpose of the document.

7. conceptos_clave : 1 example
   The response must list key concepts with one-line definitions.

8. reduccion_extrema : 1 example
   The response must select exactly 5 keywords that capture the essence.

9. conexiones_internas : 1 example
   The response must explain how the ideas relate to each other.

NOTE:
If the document is too short for resumen_3_niveles, replace it with
another idea_central.
Always produce exactly 10 examples.

STRICT RULES

1. Use ONLY information explicitly written in the document.
2. DO NOT invent numbers, facts, or details.
3. DO NOT assume missing information.
4. DO NOT use external knowledge.
5. If information is not present, DO NOT create it.
6. All questions and responses MUST be in Spanish.
7. Adapt the wording naturally — do not repeat the exact same question every time.

OUTPUT FORMAT (VERY STRICT)

You MUST return ONLY plain text using the following structure.
Do NOT return JSON.
Do NOT include explanations.
Do NOT include comments.
Do NOT include notes.

You MUST generate different types of instructions.
Do NOT repeat instruction types.
Use a mix of:
- definition
- extraction
- classification
- reasoning
- legal interpretation
- obligations
- rights
- entities
- dates
- conditions
- prohibitions
- scope of law

You MUST extract information ONLY from the document.
You are NOT allowed to infer, assume, or add information.

IMPORTANT:
Do NOT copy long parts of the document.
Responses must be compressed, synthesized, and rewritten.
Do NOT quote the document unless strictly necessary.
Each response must be shorter than the original document.

FORMAT:

You MUST follow the format EXACTLY.
If you do not follow the format, your answer is WRONG.

Do NOT write introductions.
Do NOT write explanations.
Do NOT write any text before "Ejemplo 1".
Do NOT write any text after "Ejemplo 10".

You MUST write EXACTLY this format:

Ejemplo 1:
Instruction: ...
Context:
Response: ...

Ejemplo 2:
Instruction: ...
Context:
Response: ...

Repeat until Ejemplo 10.

Your answer will be parsed by a computer program using regex.
If you change the format, the program will fail.

VALIDATION (INTERNAL)

Before answering:
- Verify all answers come from the document
- Verify exactly 10 examples are generated
- Verify the format is respected
- Verify context is always empty

LEGAL DOCUMENT

{context}

Generate the examples now.

[/INST]"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=3000,
        temperature=0.2,
        do_sample=True,
        top_p=0.3,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated = output[0][inputs["input_ids"].shape[1] :]

    result = tokenizer.decode(generated, skip_special_tokens=True)

    print("\n================ RAW MODEL OUTPUT ================\n")
    print(result)
    print("\n==================================================\n")

    return result


def process_documents():

    processed_count = 0
    max_files = 2  # Changed to 1 for debugging

    for file in os.listdir(LAWS_FOLDER):
        if not file.endswith(".txt"):
            continue

        if processed_count >= max_files:
            print(f"Límite de {max_files} archivos alcanzado")
            break

        path = LAWS_FOLDER / file

        print(f"Processing {file} ({processed_count + 1}/{max_files})")

        with open(path, "r", encoding="utf-8") as f:
            text = f.read()

        # extraer solo el contenido legal, sin metadatos
        if "CONTENIDO:" in text:
            text = text.split("CONTENIDO:", 1)[1].strip()

        # evitar contextos demasiado largos
        text = text[:4000]

        try:
            output = generate_questions(text)

            # Parsear texto -> estructura
            parsed = parse_examples(output)

            # Validar calidad
            parsed = validate_examples(parsed)

            # DEBUG
            print("Examples detected:", len(parsed["data"]))

            for i, ex in enumerate(parsed["data"]):
                print(f"\n--- Example {i + 1} ---")
                print("Instruction:", ex["instruction"][:100])
                print("Response:", ex["response"][:120])

            # Guardar si hay 10 ejemplos válidos
            if len(parsed["data"]) == 10:
                dataset["data"].extend(parsed["data"])
                processed_count += 1
                print(f"Successfully processed {file}")
            else:
                print(f"Error: Expected 10 examples, got {len(parsed['data'])}")
                print("First 500 chars of output:")
                print(output[:500])

        except Exception as e:
            print(f"Unexpected error for {file}: {e}")


def save_dataset():
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)
    print("Dataset saved:", OUTPUT_FILE)
    print("Total samples:", len(dataset["data"]))


if __name__ == "__main__":
    process_documents()
    save_dataset()

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", model.device)

CUDA available: True
Device: cuda:0
